In [19]:
# Books to Scrape 미션 해설


# crawl_test => 독립적인 모듈
# jupyter 뿐만이 아니라 vscode, cursor, pycharm, eclipse도 쓸수도 있음.
# jupyter는 셀 중심으로 움직이기 때문에 편함
# 통합개발환경이나 웹 에디터(vscode,cursor 등)를 쓰는 경우는 파일을 가져와서 써야함 
# -> import crawl_test를 쓰고난 뒤 특정 함수의 실행여부와 상관없이 모듈이 바로 실행되어버림

# div : 영역을 묶는 박스
# ol : order list <-> ul : unorder list / -. => 1
# 해당 url 안의 ol 안에는 li태그들이 있음. 이 li태그들을 order list로 순번을 가지고 있는 목록패키지를 만듦.

import re, time
import requests  # requests는 혼자 써주는 것이 관례라서 위처럼 합쳐서 쓰는 것보다는 따로 쓰는것이 나음
from bs4 import BeautifulSoup
from urllib.parse import urljoin


BASE = "https://books.toscrape.com/"

# Refactoring = 한 개의 긴 문장 코드를 기능별 여러개의 코드로 나누어서 작성 및 관리하는 프로그래밍 기법
# 한 개의 긴 문장의 코드인 경우 너무 길다보니 버그가 발생했을 때 어디에서 어떤 버그가 생겼는지 찾기 어려움. -> 코드를 보기 편하게 하기 위한 방법 (리팩토링)
# 코드 한개의 실행을 위해서 사용해야 하는 리스크가 높음. => 코드를 분산 => 메모리 스레드 => 멀티스레드 & 싱글스레드를 이해하려면 운영체제를 이해해야함. (투머치여요)

def parse_book(card, category) : 
    title = card.h3.a["title"].strip()
    product_url = urljoin(BASE, card.h3.a["href"])
    price_text = card.select_one(".price_color").get_text(strip=True)
    price_num = re.sub(r'[^0-9\.]',"",price_text)   # r'' => 안의 숫자, 문자들은 순수하게 그 숫자와 문자 자체로 인식하도록 함. (다른 연산자 등과 혼동되지 않도록 해줌)
    price = float(price_num) if price_num else None   # float() => price를 실수자료형으로 형변환시켜주는 함수
    stock_text = card.select_one(".availability").get_text(strip=True)  
    # "strip = True" => <a> " test " </a> => <a>"test"</a>와 escape sequance(숨어있는 \n)까지는 없애주지만 문자열 사이의 공백까지 없애주진 않음 "IN stock" => "IN stock
    stock = "In stock" in stock_text
    rating_word = card.select_one(".star-rating")["class"][1]
    rating_map = {"One": 1, "Two" : 2, "Three" : 3, "Four" : 4, "Five" : 5}
    rating = rating_map.get(rating_word)

    return {
        "title" : title,
        "price" : price,
        "stock" : stock,
        "rating" : rating,
        "category" : category,
        "product_url" : product_url
    }
     

# 복수의 페이지를 크롤링해오는 역할
def crawl_category(cat_url, max_pages = 2) :
    # 미스터리 > 11개 책 정보 크롤링 > 리스트 자료구조 > 리스트[]
    data =[]
    next_url = cat_url
    # 함수의 인자값으로 들어온 요소들은 함수의 실행문에서 언제든지 쓰일 수 있다! (*max_pages)
    for _ in range(max_pages) :
        html = requests.get(next_url, timeout = 10).text 
        soup = BeautifulSoup(html, "html.parser")
        category = soup.select_one(".page-header.action h1")  # <h1>Sequantial Art</h1>
        category = category.get_text(strip=True) if category else "Unknown"  # 3항 조건 연산자 (=조건문(if 조건 ~ else ~)을 한 줄로 쓰는 방법)
        for card in soup.select("ol.row > li.col-xs-6.col-sm-4.col-md-3.col-lg-3") :
            data.append(parse_book(card, category))
        nxt = soup.select_one("li.next a")
        if not nxt :
            break
        next_url = urljoin(next_url, nxt["href"])
        time.sleep(0.5)
    return data
    

    

def crawl_few_categories() :     
    # test01 = requests.get(BASE)
    # test02 = requests.get(BASE, timeout = 10).text
    html = requests.get(BASE, timeout = 10).text   # 하나로 쓸 수도 있고 두 개로 쓸 수도 있다(?)
    soup = BeautifulSoup(requests.get(BASE, timeout = 10).text, "html.parser")
    # urljoin() -> BASE url주소와 a 변수 안에 있는 주소를 합침
    cat_links = [urljoin(BASE, a["href"]) for a in soup.select(".side_categories > ul > li > ul > li > a")[:3]]

    all_rows = []
    for link in cat_links :
        all_rows.extend(crawl_category(link))
        # list 안에 list 형태인 2차원 배열로 가져올거냐 ([[a],[b],[c]]) -> append()
        # 값을 하나하나 차곡차곡 가져올거냐 ([a,b,c])-> extend()
    return all_rows


if __name__ == "__main__" :
    rows = crawl_few_categories()
    print(f"Collected : {len(rows)} rows")
    print(rows)
# 자체 실행인지, 외부에서 땡겨서 실행하고 있는지 알 수 있음.(?)
# main이라고 지정한 자체 생성파일일 경우에만 실행하라는 조건문임 

Collected : 69 rows
[{'title': "It's Only the Himalayas", 'price': 45.17, 'stock': True, 'rating': 2, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/its-only-the-himalayas_981/index.html'}, {'title': 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', 'price': 49.43, 'stock': True, 'rating': 4, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/full-moon-over-noahs-ark-an-odyssey-to-mount-ararat-and-beyond_811/index.html'}, {'title': 'See America: A Celebration of Our National Parks & Treasured Sites', 'price': 48.87, 'stock': True, 'rating': 3, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/see-america-a-celebration-of-our-national-parks-treasured-sites_732/index.html'}, {'title': 'Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel', 'price': 36.94, 'stock': True, 'rating': 2, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/vagabonding-an-uncommon-guide-to-the-art-of-long-term-w

In [12]:
list_test = [1, 2, 3, 4]
print(dir(list_test))
# 출력값의 "__" -> 속성값과 함께 만들어짐. 이터러블한 속성이 있음. 
# 하나의 셀이 모듈이 되는 순간 __name__이란 속성값을 부여받게 됨.

['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


In [ ]:
from pymongo import MongoClient

def save_mongo(rows, db_name="playground", col_name="books") :
    client = MongoClient("mongodb://localhost:27017")
    col = client[db_name][col_name]

    for doc in rows :
        col.update_one(
            {"title" :  doc["title"], "product_url" : doc["product_url"]}
        )
    print(f"[MongoDB] : {len(rows)} rows")
    